# EloRouter - Training and Inference

This notebook demonstrates how to use **EloRouter** for training and inference.

## Overview

EloRouter uses Elo ratings to select the best LLM. It's an inference-only router
that doesn't require training - it uses pre-computed Elo ratings from benchmarks.

**Key Features**:
- No training required
- Uses established Elo rating system
- Simple and interpretable
- Based on pairwise comparisons

**Note**: EloRouter always selects the highest-rated LLM based on Elo scores.

## 1. Environment Setup

In [1]:
# Install required packages (for Colab)
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install transformers torch


Cloning into 'LLMRouter'...
remote: Enumerating objects: 6017, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 6017 (delta 105), reused 96 (delta 90), pack-reused 5837 (from 2)
Receiving objects: 100% (6017/6017), 89.41 MiB | 56.31 MiB/s, done.
Resolving deltas: 100% (2945/2945), done.
Updating files: 100% (288/288), done.
/home/zhongjie/LLMRouter
Obtaining file:///home/zhongjie/LLMRouter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llmrouter-lib (pyproject.toml) ... done
  Created wheel for llmrouter-lib: filename=llmrouter_lib-0.2.0-0.editable-py3-none-any.whl size=15709 sha256=c7c85cdb9449fdbfc511504249da8139073438920eebae9193ededb8cecfcdc3
  Stored in directory: /tmp/pip-ephem-wheel-cache-5croafgh/wheels/82/4a/fd/59c4aec93c356c380

In [2]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [11]:
from llmrouter.models.elorouter import EloRouter, EloRouterTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

Environment setup complete!


## 2. Configuration

In [4]:
import yaml

CONFIG_PATH = "configs/model_config_train/elorouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

Current Configuration:
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  llm_embedding_data: data/example_data/llm_candidates/default_llm_embeddings.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  query_data_train: data/example_data/query_data/default_query_train.jsonl
  query_embedding_data: data/example_data/routing_data/query_embeddings_longformer.pt
  routing_data_test: data/example_data/routing_data/default_routing_test_data.jsonl
  routing_data_train: data/example_data/routing_data/default_routing_train_data.jsonl
metric:
  weights:
    cost: 0
    llm_judge: 0
    performance: 1
model_path:
  ini_model_path: ''
  save_model_path: saved_models/elorouter/elorouter.pkl



## 3. Load Router

In [5]:
router = EloRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of LLM candidates: {len(router.llm_data)}")
print(f"LLM candidates: {list(router.llm_data.keys())}")

✅ MetaRouter initialized successfully (YAML + data loaded).
[EloRouter] Initialized with 9 models.
Router initialized successfully!
Number of LLM candidates: 7
LLM candidates: ['qwen2.5-7b-instruct', 'llama-3.1-8b-instruct', 'mistral-7b-instruct-v0.3', 'llama-3.3-nemotron-super-49b-v1', 'llama3-70b-instruct', 'mixtral-8x7b-instruct-v0.1', 'mixtral-8x22b-instruct-v0.1']


## 4. Training

In [12]:
# Initialize trainer
trainer = EloRouterTrainer(router=router, device="cpu")

print("Trainer initialized!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Save path: {trainer.save_model_path}")

[EloRouterTrainer] Initialized.
Trainer initialized!
Number of training samples: 50544
Save path: /home/zhongjie/LLMRouter/llmrouter/saved_models/elorouter/elorouter.pkl


In [13]:
# Train the model
print("Starting EloRouter training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed successfully!")

Starting EloRouter training...


/opt/conda/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Created directory: /home/zhongjie/LLMRouter/llmrouter/saved_models/elorouter
Successfully saved pickle model: /home/zhongjie/LLMRouter/llmrouter/saved_models/elorouter/elorouter.pkl
[EloRouterTrainer] Saved Elo scores to: /home/zhongjie/LLMRouter/llmrouter/saved_models/elorouter/elorouter.pkl
Training completed successfully!


In [18]:
from llmrouter.utils import load_model

# 训练完成后加载 Elo scores
elo_scores = load_model(trainer.save_model_path)

print("LLM Elo Ratings:")
print("=" * 50)
for model, rating in sorted(elo_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model:30} {rating:.2f}")

Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/saved_models/elorouter/elorouter.pkl
LLM Elo Ratings:
  qwen2.5-7b-instruct            1346.09
  gemma-2-9b-it                  1213.53
  llama-3.1-8b-instruct          1148.56
  codegemma-7b                   1014.94
  llama-3.1-nemotron-51b-instruct 953.59
  llama-3.3-nemotron-super-49b-v1 881.35
  llama3-chatqa-1.5-8b           875.05
  mistral-7b-instruct-v0.3       867.38
  llama3-chatqa-1.5-70b          699.50


## 5. Understanding Elo Ratings

Elo rating system:
- Higher rating = better model
- Ratings updated based on pairwise comparisons
- Used in Chatbot Arena leaderboard

In [19]:
# Display Elo ratings for each LLM
print("LLM Elo Ratings:")
print("=" * 50)

# Note: Elo ratings should be available in the router or LLM data
if hasattr(router, 'elo_ratings'):
    for model, rating in sorted(router.elo_ratings.items(), key=lambda x: x[1], reverse=True):
        print(f"  {model:30} {rating}")
else:
    print("Elo ratings are computed from benchmark data.")
    print("The router will select the model with highest average performance.")

LLM Elo Ratings:
Elo ratings are computed from benchmark data.
The router will select the model with highest average performance.


## 6. Query Routing

In [15]:
EXAMPLE_QUERIES = [
    {"query": "What is the capital of France?"},
    {"query": "Solve the equation: 2x + 5 = 15"},
    {"query": "Write a Python function to check if a number is prime."},
    {"query": "Explain quantum computing in simple terms."},
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

Routing Results:
Successfully loaded pickle model: /home/zhongjie/LLMRouter/llmrouter/saved_models/elorouter/elorouter.pkl
[EloRouter] Loaded Elo scores from /home/zhongjie/LLMRouter/llmrouter/saved_models/elorouter/elorouter.pkl
1. What is the capital of France?...
   Routed to: qwen2.5-7b-instruct
2. Solve the equation: 2x + 5 = 15...
   Routed to: qwen2.5-7b-instruct
3. Write a Python function to check if a number is pr...
   Routed to: qwen2.5-7b-instruct
4. Explain quantum computing in simple terms....
   Routed to: qwen2.5-7b-instruct


## 7. Note on EloRouter Behavior

EloRouter is a **static** router - it always selects the same LLM (the highest-rated one)
regardless of the query content. This is useful as:
- A baseline for comparison
- When you want the overall best model
- When query-specific routing isn't necessary

In [16]:
# Verify static behavior
results = [router.route_single(q)['model_name'] for q in EXAMPLE_QUERIES]
all_same = len(set(results)) == 1

print(f"All queries routed to same model: {all_same}")
if all_same:
    print(f"Selected model: {results[0]}")

All queries routed to same model: True
Selected model: qwen2.5-7b-instruct


## 8. File-Based Inference

Load queries from a file and save results.

In [17]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/elorouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: data/example_data/query_data/default_query_test.jsonl
Routed 10 queries
Results saved to: outputs/elorouter_results.jsonl

Sample results:
  1. Q: There are 4 houses in a row, numbered... -> qwen2.5-7b-instruct
  2. Q: There are 3 houses in a row, numbered... -> qwen2.5-7b-instruct
  3. Q: There are 3 houses in a row, numbered... -> qwen2.5-7b-instruct


## Summary

In this notebook, we:

1. **Loaded EloRouter**: No training required
2. **Understood Elo System**: Rating-based model selection
3. **Performed Routing**: Static selection of best-rated model

**Key Takeaways**:
- EloRouter is the simplest router (no training)
- Always selects the highest-rated model
- Useful as a baseline or when simplicity is preferred

**When to use EloRouter**:
- As a baseline for comparison
- When you always want the "best" model
- When computational resources are limited

**When NOT to use EloRouter**:
- When query-specific routing is important
- When cost optimization is needed
- When different queries need different model capabilities